# Challenge 04 - Speaker Recognition (GMM)

Recognition of the speaker (31 people) of each audio track using only its MFCC frames and clustering models: one Gaussian Mixture Model per speaker, and each track is assigned to the speaker whose GMM gives the highest mean log-likelihood. Classifiers are not allowed. The predictions are saved in `Es6Predictions.txt`.

## Setup

In [1]:
import os
os.environ['LOKY_MAX_CPU_COUNT'] = '4'

import numpy as np
from sklearn.mixture import GaussianMixture

# Configuration
db_path = '../../DBs/AudioDigits_ML26_27'
train_file = 'audio_ml2627_train.pkl'
test_file = 'audio_ml2627_test_no_labels.pkl'
result_path = '../Es6Predictions.txt'

# GMM hyperparameters
n_components = 8
covariance_type = 'diag'
random_state = 1234

## Helper Functions

Dataset loading, training of one GMM per speaker, speaker prediction by mean log-likelihood, and the export of the predictions.

In [2]:
def load_dataset(db_path, filename):
    """Pickled dictionary of aligned lists (mfccs, person_ids, session_ids, ...)."""
    return np.load(os.path.join(db_path, filename), allow_pickle=True)


def preprocess(mfccs):
    """Per-track preprocessing of the MFCC frames (identity for now)."""
    return [m for m in mfccs]


def train_speaker_models(mfccs, speakers):
    """Fit one GMM per speaker on the concatenated frames of all its tracks."""
    models = {}
    for speaker in np.unique(speakers):
        frames = np.vstack([m for m, s in zip(mfccs, speakers) if s == speaker])
        models[speaker] = GaussianMixture(n_components=n_components, covariance_type=covariance_type,
                                          random_state=random_state).fit(frames)
    return models


def predict_speaker(models, mfccs):
    """Speaker whose GMM has the highest mean log-likelihood (gmm.score) on each track."""
    speakers = list(models)
    scores = np.array([[models[s].score(m) for s in speakers] for m in mfccs])
    return np.array(speakers)[scores.argmax(axis=1)]


def save_predictions(predictions, path):
    """One integer speaker id per line, no header."""
    np.savetxt(path, np.asarray(predictions, dtype=int), fmt='%i')
    print(f'Predictions successfully saved to {path}')

## Data Loading

The training set contains the tracks of sessions 1 and 4 (620 tracks, 20 per speaker), the test set those of sessions 2 and 3 (620 tracks, without labels). Only the MFCCs are used: `features` and `digits` are not allowed.

In [3]:
train_data = load_dataset(db_path, train_file)
test_data = load_dataset(db_path, test_file)

train_x = preprocess(train_data['mfccs'])
train_y = np.array(train_data['person_ids'])
train_sessions = np.array(train_data['session_ids'])
test_x = preprocess(test_data['mfccs'])

print('Training tracks: %d, test tracks: %d, speakers: %d' % (len(train_x), len(test_x), len(np.unique(train_y))))
print('MFCC frames per track: %d-%d' % (min(len(m) for m in train_x), max(len(m) for m in train_x)))

Training tracks: 620, test tracks: 620, speakers: 31
MFCC frames per track: 79-745


## Model: One GMM per Speaker

Baseline: one `GaussianMixture` per speaker, without any additional preprocessing of the MFCCs. To improve: preprocessing (standardization, deltas), number of components, covariance type.

## Model Evaluation

The split is done by session, to simulate new recording conditions: the GMMs are trained on one session and validated on the other, in both directions.

In [4]:
for train_session, val_session in [(1, 4), (4, 1)]:
    tr = train_sessions == train_session
    val = train_sessions == val_session
    models = train_speaker_models([m for m, t in zip(train_x, tr) if t], train_y[tr])
    val_pred = predict_speaker(models, [m for m, v in zip(train_x, val) if v])
    print('Train session %d -> validation session %d: accuracy %.4f' % (train_session, val_session, np.mean(val_pred == train_y[val])))

Train session 1 -> validation session 4: accuracy 0.5355
Train session 4 -> validation session 1: accuracy 0.4645


## Prediction and Export

The GMMs are retrained on all training tracks (sessions 1 and 4) and each test track is assigned to the most likely speaker. The predictions are saved in `Es6Predictions.txt`, one speaker id per line, in the order of the test set.

In [5]:
models = train_speaker_models(train_x, train_y)
predictions = predict_speaker(models, test_x)
assert len(predictions) == 620
save_predictions(predictions, result_path)

Predictions successfully saved to ../Es6Predictions.txt
